In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tabml.datasets

In [4]:
df = tabml.datasets.download_movielen_1m()
df.keys()

dict_keys(['users', 'movies', 'ratings'])

In [34]:
users, ratings, movies = df['users'], df['ratings'], df['movies']
print('Number of user: ', len(users['UserID']))
print('Number of movie: ', len(movies['MovieID']))

Number of user:  6040
Number of movie:  3883


In [35]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6040 entries, 0 to 6039
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   UserID      6040 non-null   int64 
 1   Gender      6040 non-null   object
 2   Age         6040 non-null   int64 
 3   Occupation  6040 non-null   int64 
 4   Zip-code    6040 non-null   object
dtypes: int64(3), object(2)
memory usage: 236.1+ KB


In [36]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   MovieID  3883 non-null   int64 
 1   Title    3883 non-null   object
 2   Genres   3883 non-null   object
dtypes: int64(1), object(2)
memory usage: 91.1+ KB


In [37]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   UserID     1000209 non-null  int64
 1   MovieID    1000209 non-null  int64
 2   Rating     1000209 non-null  int64
 3   Timestamp  1000209 non-null  int64
dtypes: int64(4)
memory usage: 30.5 MB


In [30]:
# CREATING UTILITY MATRIX: USER-ITEM, ITEM-RATING.

In [38]:
# user_index_by_id = {id: idx for id,idx in enumerate(users['UserID'])}
# movie_index_by_id = {id: idx for id, idx in enumerate(movies['MovieID'])}


# print(len(user_index_by_id))
# print(len(movie_index_by_id))

6040
3883


In [39]:
genre_list = []
for genres in movies['Genres'].unique():
    for genre in genres.split('|'):
        if  genre not in genre_list:
            genre_list.append(genre)

genre_list

['Animation',
 "Children's",
 'Comedy',
 'Adventure',
 'Fantasy',
 'Romance',
 'Drama',
 'Action',
 'Crime',
 'Thriller',
 'Horror',
 'Sci-Fi',
 'Documentary',
 'War',
 'Musical',
 'Mystery',
 'Film-Noir',
 'Western']

In [40]:
genre_index_by_name = {name:i for i, name in enumerate(genre_list)}
movie_features = np.zeros((len(movies), len(genre_list)))

for idx, movie_genres in enumerate(movies['Genres']):
    for genre in movie_genres.split('|'):
        genre_idx = genre_index_by_name[genre]
        movie_features[idx, genre_idx] = 1

movie_features

array([[1., 1., 1., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [23]:
ratings[ratings['UserID'] == 1]

,UserID,MovieID,Rating,Timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
5,1,1197,3,978302268
6,1,1287,5,978302039
7,1,2804,5,978300719
8,1,594,4,978302268
9,1,919,4,978301368


In [51]:
from tqdm import tqdm

movie_index_by_id = [movie for movie in movies['MovieID']]
user_index_by_id = [user for user in users['UserID']]

user_item = np.zeros((len(user_index_by_id), len(movie_index_by_id)))

for user_id in tqdm(ratings['UserID'].unique()):
    for movie_id in ratings[ratings['UserID'] == user_id]['MovieID']:
        idx = movie_index_by_id.index(movie_id)
        user_item[user_id, idx] = ratings[ratings['UserID'] == user_id][ratings['MovieID'] == movie_id]['Rating']
user_item

  0%|          | 0/6040 [00:00<?, ?it/s]/tmp/ipykernel_119176/2983363962.py:11: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  user_item[user_id, idx] = ratings[ratings['UserID'] == user_id][ratings['MovieID'] == movie_id]['Rating']
 18%|█▊        | 1111/6040 [08:41<25:59,  3.16it/s]  

In [55]:
# Similarity matrix: similarity between users on a specific item.

def similarity(vect1, vect2, type='cosine'):
    if type == None or type == 'cosine':
        return np.cos(vect1, vect2)
    if type == 'pearson':
        return np.corrcoef(vect1, vect2)[0, 1]

array([[ 53.,   7.,   6., ...,   0.,  15.,  17.],
       [  7., 129.,  12., ...,   3.,   8.,  47.],
       [  6.,  12.,  51., ...,   3.,   8.,  20.],
       ...,
       [  0.,   3.,   3., ...,  20.,   7.,  11.],
       [ 15.,   8.,   8., ...,   7., 123.,  41.],
       [ 17.,  47.,  20., ...,  11.,  41., 341.]])

In [78]:
user_neighbor={}
for user in tqdm(list(user_index_by_id.keys())):
    similarity_user_list = similarity_matrix[user, :]
    user_neighbor[user] = np.argpartition(similarity_user_list, len(similarity_user_list)//2)[-30:]

user_neighbor


100%|██████████| 6040/6040 [00:00<00:00, 10963.02it/s]


{0: array([6010,  148, 6012,  147,  146, 6015,  145,  138,  135,  130,  122,
         116,  113,   95,   91,   74,   61,   58,   57,   52,   47,   43,
          35,   32,   25, 6035,   18,   17,    9,    0]),
 1: array([6010,   48,   47,   43,   41, 6015,   35, 6017,   34,   32, 6020,
          28, 6022, 6023,   25,   23,   22,   21,   18,   17,   16,   14,
          12,   10, 6034, 6035, 6036,    9,    7, 6039]),
 2: array([6010,   17, 2339,   14, 2222, 6015,   12, 2224, 2333, 2230, 6020,
        2322, 2241, 2243, 2254, 2257,    9, 2303, 2302, 2301, 2261, 2262,
        2263, 2264, 2287, 6035, 2270,    2, 2278, 6039]),
 3: array([6010,   18,   17, 2889,   16, 6015, 2942, 6017, 2891, 2940, 6020,
        2893, 2894, 6023,    9, 2895, 2933, 2897, 2898, 2928, 2927, 2899,
        6032, 2900, 2917, 6035, 2906,    3, 2908, 6039]),
 4: array([  58,   57, 6012,   55,   52, 6015,   47, 6017,   44,   41,   38,
          35,   32,   25, 6024,   22,   21,   18,   17,   16,   14,   10,
           9,